# RAG 지식베이스 구축
이형 유튜브 텍스트 → ChromaDB 벡터 저장소

**로컬/코랩 공용** — 경로만 바꾸면 코랩에서도 동작합니다.

In [ ]:
%pip install "chromadb==0.6.3" sentence-transformers

     ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
     ---------------------------------------- 0.2/23.5 MB 5.9 MB/s eta 0:00:04
     ---------------------------------------- 0.3/23.5 MB 2.6 MB/s eta 0:00:09
      --------------------------------------- 0.3/23.5 MB 2.2 MB/s eta 0:00:11
      --------------------------------------- 0.4/23.5 MB 2.1 MB/s eta 0:00:11
     - -------------------------------------- 0.7/23.5 MB 2.4 MB/s eta 0:00:10
     - -------------------------------------- 1.0/23.5 MB 2.9 MB/s eta 0:00:08
     -- ------------------------------------- 1.7/23.5 MB 4.6 MB/s eta 0:00:05
     ---- ----------------------------------- 2.4/23.5 MB 5.4 MB/s eta 0:00:04
     ----- ---------------------------------- 3.4/23.5 MB 6.7 MB/s eta 0:00:04
     ------- -------------------------------- 4.2/23.5 MB 7.4 MB/s eta 0:00:03
     -------- ------------------------------- 5.1/23.5 MB 8.4 MB/s eta 0:00:03
     ---------- ----------------------------- 6.1/23.5 MB 9

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.35.1 which is incompatible.
mediapipe 0.10.9 requires protobuf<4,>=3.11, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from pathlib import Path

# ─── 경로 설정 ───────────────────────────────────────────────
# 코랩 사용 시: BASE = Path('/content/drive/MyDrive/프로젝트3(면접)')
BASE = Path(r'C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)')

# 텍스트 소스 폴더 (애매 폴더 제외 — 면접과 무관한 내용 포함)
TXT_DIRS = [
    BASE / '딥러닝' / '이형 유튜브 영상' / '원본' / 'output_txt',
    BASE / '딥러닝' / '이형 유튜브 영상' / '추가' / 'output_txt',
    BASE / '딥러닝' / '이형 유튜브 영상' / '피드백편집본' / 'output_txt',
]

# ChromaDB 저장 위치
CHROMA_DIR = BASE / 'chroma_db'
CHROMA_DIR.mkdir(exist_ok=True)

print('경로 확인:')
for d in TXT_DIRS:
    count = len(list(d.glob('*.txt'))) if d.exists() else 0
    print(f'  {d.name:<20} → txt {count}개  (존재: {d.exists()})')

경로 확인:
  output_txt           → txt 6개  (존재: True)
  output_txt           → txt 23개  (존재: True)
  output_txt           → txt 4개  (존재: True)


In [ ]:
# ─── txt 파일 전체 로드 ──────────────────────────────────────
def load_txt_files(dirs):
    docs = []
    for d in dirs:
        for f in sorted(d.glob('*.txt')):
            text = f.read_text(encoding='utf-8', errors='ignore').strip()
            if text:
                docs.append({'source': f.stem, 'text': text})
    return docs

docs = load_txt_files(TXT_DIRS)
print(f'총 {len(docs)}개 문서 로드')
for d in docs[:3]:
    print(f'  [{d["source"][:30]}]  {len(d["text"])}자')

총 33개 문서 로드
  [“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한]  53770자
  [공기업 공공기관 면접은 뭐가 다를까]  60038자
  [남자 33세 4년 공백기 퇴직사유 경력 지원자를 보는 ]  58719자


In [ ]:
# ─── 텍스트 청킹 ─────────────────────────────────────────────
# 한국어는 문장 단위로 자르되 chunk_size 글자씩, overlap으로 문맥 유지
import re

def split_sentences(text):
    # 마침표/물음표/느낌표 기준 분리
    return [s.strip() for s in re.split(r'(?<=[.?!])\s+', text) if len(s.strip()) > 10]

def chunk_text(text, source, chunk_size=300, overlap=50):
    sentences = split_sentences(text)
    chunks = []
    buf, buf_len = [], 0

    for sent in sentences:
        buf.append(sent)
        buf_len += len(sent)
        if buf_len >= chunk_size:
            chunks.append({'text': ' '.join(buf), 'source': source})
            # overlap: 마지막 문장 유지
            overlap_sents = []
            ol = 0
            for s in reversed(buf):
                if ol + len(s) > overlap:
                    break
                overlap_sents.insert(0, s)
                ol += len(s)
            buf, buf_len = overlap_sents, ol

    if buf:
        chunks.append({'text': ' '.join(buf), 'source': source})
    return chunks

all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_text(doc['text'], doc['source']))

print(f'총 {len(all_chunks)}개 청크 생성')
print(f'평균 청크 길이: {sum(len(c["text"]) for c in all_chunks) // len(all_chunks)}자')
print('\n샘플 청크:')
print(all_chunks[0]['text'][:200])

총 786개 청크 생성
평균 청크 길이: 587자

샘플 청크:
[0.00초 ~ 2.62초] 그러면 스타트업에서 그런 좋은 기억 있으신데
    (0.00 ~ 0.26) 그러면
    (0.26 ~ 1.16) 스타트업에서
    (1.16 ~ 1.46) 그런
    (1.46 ~ 1.72) 좋은
    (1.72 ~ 1.92) 기억
    (1.92 ~ 2.62) 있으신데

[2.62초 ~ 4.86초] 계속 스타트업 쪽


In [ ]:
# ─── 임베딩 모델 로드 ────────────────────────────────────────
# paraphrase-multilingual: 한국어 포함 50개 언어 지원, 384차원
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('임베딩 모델 로드 완료')
print(f'벡터 차원: {model.get_sentence_embedding_dimension()}')

c:\Users\82105\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2362.83it/s]


임베딩 모델 로드 완료
벡터 차원: 384


C:\Users\82105\AppData\Local\Temp\ipykernel_17216\1038816666.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'벡터 차원: {model.get_sentence_embedding_dimension()}')


In [ ]:
# ─── ChromaDB에 저장 ─────────────────────────────────────────
import chromadb
from chromadb.utils import embedding_functions

# 로컬 디스크에 영구 저장
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# 기존 컬렉션 있으면 삭제 후 재생성 (재실행 시 중복 방지)
try:
    client.delete_collection('interview_rag')
    print('기존 컬렉션 삭제')
except:
    pass

# sentence-transformers 임베딩 함수 연결
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = client.create_collection('interview_rag', embedding_function=ef)

# 배치로 나눠서 저장 (메모리 관리)
BATCH = 100
for i in range(0, len(all_chunks), BATCH):
    batch = all_chunks[i:i+BATCH]
    collection.add(
        documents=[c['text'] for c in batch],
        metadatas=[{'source': c['source']} for c in batch],
        ids=[f'chunk_{i+j}' for j, _ in enumerate(batch)]
    )
    print(f'  {i+len(batch)}/{len(all_chunks)} 저장 완료')

print(f'\nChromaDB 저장 완료 → {CHROMA_DIR}')
print(f'총 {collection.count()}개 벡터 저장됨')

ImportError: cannot import name 'Sentinel' from 'typing_extensions' (c:\Users\82105\AppData\Local\Programs\Python\Python310\lib\site-packages\typing_extensions.py)

In [ ]:
# ─── 검색 테스트 ──────────────────────────────────────────────
def search(query, n=3):
    results = collection.query(query_texts=[query], n_results=n)
    print(f'쿼리: "{query}"\n')
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        print(f'[{i+1}] 출처: {meta["source"]}')
        print(f'     {doc[:150]}...\n')

# 테스트
search('1분 자기소개 잘 하는 방법')
search('면접관이 싫어하는 지원자 유형')
search('지원동기 답변 방법')

## 완료
- ChromaDB가 `chroma_db/` 폴더에 영구 저장됩니다
- 다음 단계: `02_video_analysis.ipynb` — 영상 표정/자세 분석

### 코랩으로 이전 시
```python
# 상단 BASE 경로만 변경
BASE = Path('/content/drive/MyDrive/프로젝트3(면접)')
```
chroma_db 폴더째로 Drive에 올리면 재구축 불필요